# tui

> Terminal live inspector — a tight, gruvbox-dark Textual frontend that runs in
> its own terminal and streams the live kernel namespace from the in-kernel paar
> server (the same one `paar.fasthtml.inspector()` starts), the way an iframe or
> `htop` sits live in a pane.

Speaks only the server's JSON API + `/live` websocket, so it attaches to any paar process
found in the registry. `Client` transparently reads the server's owner token from
`<reg_dir>/token-<port>` (same-user file, mode 0600) and sends it as `X-Paar-Token`, so the
exec bar keeps working on token-gated servers while agent processes — which are pointed at
`/agent/api` and never handed the token path convention — stay restricted.

In [ ]:
#| default_exp tui

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio, json, argparse
from urllib.parse import quote, urlsplit
import httpx
from paar.registry import reg_dir

# Gruvbox-dark palette (https://github.com/morhetz/gruvbox) — the whole UI is keyed off these.
GRUV = dict(
    bg_h='#1d2021', bg='#282828', bg1='#3c3836', bg2='#504945', bg3='#665c54', bg4='#7c6f64',
    fg='#ebdbb2', fg2='#d5c4a1', fg4='#a89984', gray='#928374',
    red='#fb4934', green='#b8bb26', yellow='#fabd2f', blue='#83a598',
    purple='#d3869b', aqua='#8ec07c', orange='#fe8019')

def _acc(accessor):
    "URL-safe positional accessor, matching the web frontend's encoding."
    return quote(json.dumps(list(accessor)), safe='')

def _ws_url(base):
    "http(s) base -> ws(s) URL for the /live nudge socket."
    b = base.rstrip('/')
    if b.startswith('https'): return 'wss' + b[5:] + '/live'
    if b.startswith('http'):  return 'ws'  + b[4:] + '/live'
    return b + '/live'

def _read_token(base):
    "Owner token for a local server from <reg_dir>/token-<port> (written by serve()/inspector()); None when absent."
    try:
        port = urlsplit(base).port
        return (reg_dir()/f'token-{port}').read_text().strip() or None
    except Exception: return None

class Client:
    "Async HTTP+WS client for a running in-kernel paar server (see paar.fasthtml)."
    def __init__(self, base='http://127.0.0.1:8000', token=None):
        self.base = base.rstrip('/')
        self.ws_url = _ws_url(self.base)
        tok = token if token is not None else _read_token(self.base)
        self._http = httpx.AsyncClient(base_url=self.base, timeout=10.0,
                                       headers={'X-Paar-Token': tok} if tok else None)
    async def rows(self, profile=None):
        r = await self._http.get('/api/rows', params={'profile': profile} if profile else None)
        r.raise_for_status(); return r.json()
    async def expand(self, accessor, offset=0):
        r = await self._http.get(f'/api/expand?accessor={_acc(accessor)}&offset={offset}')
        r.raise_for_status(); return r.json()
    async def grid(self, accessor, roff=0, coff=0):
        r = await self._http.get(f'/api/grid?accessor={_acc(accessor)}&roff={roff}&coff={coff}')
        r.raise_for_status(); return r.json()
    async def envs(self):
        "Live paar environments known to the connected server (the /api/envs discovery feed)."
        r = await self._http.get('/api/envs'); r.raise_for_status()
        return r.json().get('envs', [])
    async def sessions(self):
        "Saved session notebooks (current first): {'current':str, 'sessions':[{name,count,current}]}."
        r = await self._http.get('/api/sessions'); r.raise_for_status(); return r.json()
    async def session(self, name):
        "Code-cell sources for a saved session: {'name':str, 'cells':[str]}."
        r = await self._http.get('/api/session', params={'name': name}); r.raise_for_status(); return r.json()
    async def complete(self, code, pos):
        "Completion candidates at cursor pos (see /complete): {'from':int, 'matches':[str]}."
        r = await self._http.get('/complete', params={'code': code, 'pos': pos})
        r.raise_for_status(); return r.json()
    async def exec_code(self, code, scope='global'):
        "Run code on the server; returns {ok, result, stdout, error} (see /api/exec)."
        r = await self._http.get('/api/exec', params={'code': code, 'scope': scope})
        r.raise_for_status(); return r.json()
    async def share(self, name):
        "Push a read-only copy of owner var `name` into the agent's isolated worker (owner /push endpoint)."
        r = await self._http.post('/push', params={'name': name}); r.raise_for_status(); return r.text
    async def share_all(self):
        "Push read-only copies of all owner vars into the agent's isolated worker (owner /push_all endpoint)."
        r = await self._http.post('/push_all'); r.raise_for_status(); return r.text
    async def aclose(self): await self._http.aclose()

In [ ]:
#| export
from rich.text import Text

def fmt_row(v, maxval=90):
    "A VarInfo dict -> a single-line gruvbox Rich Text: name = {type: shape} value  dtype ▦, mirroring the web row."
    g, err = GRUV, v.get('is_error')
    t = Text(no_wrap=True, overflow='ellipsis')
    t.append(v.get('name',''), style=f"bold {g['red'] if err else g['aqua']}")
    typ = v.get('type') or ''
    if v.get('shape'): typ = f"{typ}: {v['shape']}"
    if typ:
        t.append(' = ', style=g['gray']); t.append('{'+typ+'}', style=g['gray'])
    val = v.get('value') or ''
    if len(val) > maxval: val = val[:maxval-1] + '\u2026'
    if val: t.append('  '); t.append(val, style=g['red'] if err else g['fg2'])
    if v.get('dtype'): t.append(f"  {v['dtype']}", style=f"italic {g['gray']}")
    if v.get('qualifier') and v['qualifier'] != v.get('type'):
        t.append(f"  {v['qualifier']}", style=f"italic {g['bg4']}")   # fully-qualified type (web shows as hover)
    if v.get('is_grid'): t.append('  \u25a6', style=g['orange'])   # ▦ = openable grid panel
    return t

def group_label(name):
    "Uppercased dim-yellow heading for a category group (Modules / Functions / …)."
    return Text(str(name).upper(), style=f"bold {GRUV['yellow']}")

In [ ]:
#| export
from textual.theme import Theme

def gruvbox_theme():
    "A tightened gruvbox-dark Textual theme (deep bg, aqua cursor)."
    g = GRUV
    return Theme(
        name='paar-gruvbox', dark=True,
        primary=g['aqua'], secondary=g['blue'], accent=g['orange'],
        warning=g['yellow'], error=g['red'], success=g['green'],
        foreground=g['fg'], background=g['bg_h'], surface=g['bg'], panel=g['bg1'],
        variables={'block-cursor-foreground': g['bg_h'], 'block-cursor-background': g['aqua'],
                   'block-cursor-text-style': 'none', 'footer-key-foreground': g['aqua'],
                   'scrollbar': g['bg2'], 'scrollbar-hover': g['bg3']})

In [ ]:
#| export
from textual import work
from textual.app import App, ComposeResult
from textual.binding import Binding
from textual.containers import Vertical
from textual.screen import ModalScreen
from textual.widgets import Tree, DataTable, Static, Footer, OptionList, Input
from textual.widgets.option_list import Option

_PAGE = 50   # grid page window (rows/cols) per step

class EnvScreen(ModalScreen):
    "Overlay list of live paar environments; dismiss returns the chosen base URL (or None)."
    BINDINGS = [Binding('escape', 'dismiss_none', 'close')]
    CSS = """
    EnvScreen { align: center middle; background: $background 60%; }
    #envbox { width: 60%; max-width: 80; height: auto; max-height: 80%;
              background: $panel; border: round $primary; padding: 0 1; }
    #envtitle { color: $accent; text-style: bold; padding: 0 0 0 1; }
    #envlist { background: $panel; }
    """
    def __init__(self, envs, current=None):
        super().__init__(); self.envs = envs; self.current = current
    def compose(self):
        dot = '●'
        opts = [Option((dot if e['base']==self.current else ' ') + f"  {e['name']}   {e['base']}", id=e['base'])
                for e in self.envs]
        yield Vertical(Static('switch environment', id='envtitle'), OptionList(*opts, id='envlist'), id='envbox')
    def on_option_list_option_selected(self, event): self.dismiss(event.option.id)
    def action_dismiss_none(self): self.dismiss(None)

class CodeInput(Input):
    "Exec input that routes ↑/↓/Tab/Enter/Esc to the autocomplete popup while it is open."
    async def _on_key(self, event):
        app = self.app
        if getattr(app, '_ac_visible', False) and event.key in ('up','down','tab','enter','escape'):
            event.stop(); event.prevent_default(); app._ac_key(event.key); return
        await super()._on_key(event)

class SessionScreen(ModalScreen):
    "Overlay to browse saved session notebooks and load a past cell into the exec bar."
    BINDINGS = [Binding('escape', 'dismiss_none', 'close')]
    CSS = """
    SessionScreen { align: center middle; background: $background 60%; }
    #sessbox { width: 80%; max-width: 110; height: auto; max-height: 80%;
               background: $panel; border: round $primary; padding: 0 1; }
    #sesstitle { color: $accent; text-style: bold; padding: 0 0 0 1; }
    #sesslist { background: $panel; height: auto; max-height: 30; }
    """
    def __init__(self, client):
        super().__init__(); self.client = client; self._cells = []
    def compose(self):
        yield Vertical(Static('sessions', id='sesstitle'), OptionList(id='sesslist'), id='sessbox')
    async def on_mount(self): await self._show_sessions()
    async def _show_sessions(self):
        ol = self.query_one('#sesslist', OptionList); ol.clear_options()
        try: data = await self.client.sessions()
        except Exception as e: ol.add_option(Option(f'error: {e}')); return
        self.query_one('#sesstitle', Static).update('sessions — pick one')
        for s in data.get('sessions', []):
            mark = '● ' if s['current'] else '  '
            ol.add_option(Option(f"{mark}{s['name']}   ({s['count']})", id='S:' + s['name']))
        if not data.get('sessions'): ol.add_option(Option('(no sessions)'))
    async def _show_cells(self, name):
        ol = self.query_one('#sesslist', OptionList); ol.clear_options()
        try: data = await self.client.session(name)
        except Exception as e: ol.add_option(Option(f'error: {e}')); return
        self.query_one('#sesstitle', Static).update(f'{name} — pick a cell to load')
        ol.add_option(Option('‹ back', id='BACK'))
        self._cells = data.get('cells', [])
        for i, code in enumerate(self._cells):
            preview = ' '.join(code.split())[:80]
            ol.add_option(Option(preview or '(empty)', id=f'C:{i}'))
        if not self._cells: ol.add_option(Option('(empty session)'))
    async def on_option_list_option_selected(self, event):
        oid = event.option.id or ''
        if oid == 'BACK': await self._show_sessions()
        elif oid.startswith('S:'): await self._show_cells(oid[2:])
        elif oid.startswith('C:'): self.dismiss(self._cells[int(oid[2:])])
    def action_dismiss_none(self): self.dismiss(None)

class InspectorApp(App):
    "Live terminal inspector: a lazy gruvbox tree of the kernel namespace with a filter, code-exec bar, and paged grid, refreshed on every cell run."
    CSS = """
    Screen { background: $background; }
    #topbar { height: 1; background: $panel; color: $foreground; padding: 0 1; }
    #filter { height: 1; border: none; padding: 0 1; background: $surface; color: $foreground; }
    #filter:focus { background: $panel; }
    #tree { padding: 0 0 0 1; background: $background; scrollbar-size-vertical: 1; }
    #tree > .tree--guides          { color: $surface; }
    #tree > .tree--guides-hover    { color: $panel; }
    #tree > .tree--guides-selected { color: $accent; }
    #gridwrap { display: none; height: 45%; }
    #gridwrap.visible { display: block; }
    #gridnav { height: 1; background: $panel; color: $foreground; padding: 0 1; }
    #grid { background: $background; scrollbar-size: 1 1; }
    #execwrap { display: none; height: auto; max-height: 45%; }
    #execwrap.visible { display: block; }
    #execout { height: auto; max-height: 10; background: $background; color: $foreground; padding: 0 1; }
    #code { border: none; background: $surface; color: $foreground; }
    #code:focus { background: $panel; }
    #complist { display: none; height: auto; max-height: 12; background: $panel; color: $foreground; padding: 0 1; }
    #complist.visible { display: block; }
    Footer { background: $panel; }
    Footer > .footer-key--description { color: $foreground; }
    """
    BINDINGS = [
        Binding('r', 'refresh', 'refresh'),
        Binding('slash', 'focus_filter', 'filter'),
        Binding('x', 'toggle_exec', 'exec'),
        Binding('g', 'toggle_grid', 'grid'),
        Binding('e', 'switch_env', 'env'),
        Binding('s', 'sessions', 'sessions'),
        Binding('p', 'share_selected', 'share→agent'),
        Binding('a', 'share_all', 'share all'),
        Binding('escape', 'hide_panels', 'close', show=False),
        Binding('1', "profile('minimal')", 'min'),
        Binding('2', "profile('standard')", 'std'),
        Binding('3', "profile('full')", 'full'),
        Binding('bracketright', "grid_page('rows', 1)", 'rows+', show=False),
        Binding('bracketleft',  "grid_page('rows', -1)", 'rows-', show=False),
        Binding('right_curly_bracket', "grid_page('cols', 1)", 'cols+', show=False),
        Binding('left_curly_bracket',  "grid_page('cols', -1)", 'cols-', show=False),
        Binding('q', 'quit', 'quit'),
    ]

    def __init__(self, base='http://127.0.0.1:8000', profile='standard', client=None, connect_ws=True):
        super().__init__()
        self.client = client or Client(base)
        self.base = self.client.base
        self.env = None        # current env label (from /api/envs)
        self.profile = profile
        self.connect_ws = connect_ws
        self.filter = ''       # case-insensitive name filter
        self._data = None      # last /api/rows payload, for re-filtering without a refetch
        self._by_acc = {}      # tuple(accessor) -> TreeNode  (data nodes only)
        self._loaded = set()   # accessors whose children have been fetched
        self._live = False
        self._grid = None      # [accessor, roff, coff] while the grid panel is open
        self._ac_matches = []; self._ac_sel = 0; self._ac_from = 0
        self._ac_visible = False; self._ac_timer = None; self._ac_suppress = False

    # -- layout ---------------------------------------------------------------
    def compose(self) -> ComposeResult:
        yield Static(id='topbar')
        yield Input(placeholder='filter by name…  ( / )', id='filter')
        self._tree = Tree('paar', id='tree')
        self._tree.show_root = False
        self._tree.guide_depth = 2
        yield self._tree
        with Vertical(id='gridwrap'):
            yield Static(id='gridnav')
            yield DataTable(id='grid', zebra_stripes=True, cursor_type='row', show_row_labels=True)
        with Vertical(id='execwrap'):
            yield Static(id='execout')
            yield Static(id='complist')
            yield CodeInput(placeholder='python — Enter runs it   ( x toggles, esc closes )', id='code')
        yield Footer()

    async def on_mount(self):
        self.register_theme(gruvbox_theme()); self.theme = 'paar-gruvbox'
        await self._reload()
        self._tree.focus()
        if self.connect_ws: self.run_worker(self._ws_loop(), name='ws', exclusive=True)

    # -- tree building --------------------------------------------------------
    def _add_node(self, parent, v):
        acc = tuple(v['accessor'])
        kind = 'more' if v.get('more_offset') is not None else 'node'
        expandable = kind == 'node' and bool(v.get('is_container'))
        node = parent.add(fmt_row(v), data={'kind': kind, 'v': v, 'acc': acc},
                          allow_expand=expandable)
        if kind == 'node': self._by_acc[acc] = node
        return node

    def _add_grid_child(self, parent, v):
        "A leaf '▦ grid' child under a gridable container — selecting it opens the grid (mirrors the web grid toggle)."
        parent.add(Text('▦ grid', style=GRUV['orange']),
                   data={'kind': 'grid', 'v': v, 'acc': tuple(v['accessor'])}, allow_expand=False)

    def _match(self, v):
        "Case-insensitive name filter (mirrors the web filter bar)."
        return (not self.filter) or self.filter in (v.get('name','') or '').lower()

    def _build(self, data):
        self._tree.clear(); self._by_acc.clear(); self._loaded.clear()
        root = self._tree.root
        for grp in data['groups']:
            nodes = [v for v in grp['nodes'] if self._match(v)]
            if self.filter and not nodes: continue          # drop groups with no match
            if grp['label'] is None:
                for v in nodes: self._add_node(root, v)
            else:
                gnode = root.add(group_label(grp['label']), data={'kind': 'group'})
                for v in nodes: self._add_node(gnode, v)
                gnode.expand()

    async def _load(self, node, offset=0):
        "Fetch and append one page of a container node's children (grid toggle first, for gridable containers)."
        v0 = node.data['v']
        if offset == 0 and v0.get('is_grid'): self._add_grid_child(node, v0)
        for v in await self.client.expand(node.data['acc'], offset):
            self._add_node(node, v)
        self._loaded.add(node.data['acc'])

    def _cursor_acc(self):
        n = self._tree.cursor_node
        return (n.data or {}).get('acc') if n and n.data else None

    async def _reload(self):
        "Re-pull the namespace and rebuild, replaying open subtrees and the cursor (the live htop-style refresh)."
        try:
            data = await self.client.rows(self.profile)
        except Exception as e:
            self._set_live(False); self._topbar(err=str(e)); return
        expanded = sorted((acc for acc, n in self._by_acc.items() if n.is_expanded), key=len)
        cursor = self._cursor_acc()
        self.profile = data.get('profile', self.profile)
        self._data = data
        self._build(data)
        for acc in expanded:                      # replay drill-down, shallow first
            node = self._by_acc.get(acc)
            if node is None or not node.allow_expand: continue
            if acc not in self._loaded: await self._load(node)
            node.expand()
        if cursor and cursor in self._by_acc:
            self._tree.move_cursor(self._by_acc[cursor])
        self._topbar()

    # -- events ---------------------------------------------------------------
    async def on_tree_node_expanded(self, event):
        d = event.node.data or {}
        if d.get('kind') == 'node' and d['acc'] not in self._loaded and not event.node.children:
            await self._load(event.node)

    async def on_tree_node_selected(self, event):
        d = event.node.data or {}
        if d.get('kind') == 'more':
            v, parent = d['v'], event.node.parent
            event.node.remove()
            for nv in await self.client.expand(v['accessor'], v['more_offset']):
                self._add_node(parent, nv)
        elif d.get('kind') == 'grid':
            await self._open_grid(d['v'])
        elif d.get('kind') == 'node' and d['v'].get('is_grid') and not d['v'].get('is_container'):
            await self._open_grid(d['v'])   # pure gridable: the row itself opens the grid (containers use the grid child)

    # -- grid panel -----------------------------------------------------------
    async def _open_grid(self, v):
        self._grid = [list(v['accessor']), 0, 0]
        await self._render_grid()
        self.query_one('#gridwrap').add_class('visible')

    async def _render_grid(self):
        acc, roff, coff = self._grid
        page = await self.client.grid(acc, roff, coff)
        dt = self.query_one('#grid', DataTable); dt.clear(columns=True)
        nav = self.query_one('#gridnav', Static)
        if not page:
            nav.update('not gridable'); return
        for h in page['headers']: dt.add_column(str(h))
        for ix, row in zip(page['index'], page['cells']):
            dt.add_row(*[str(c) for c in row], label=str(ix))
        npr, npc = len(page['index']), len(page['headers'])
        nav.update(f"[b {GRUV['orange']}]▦[/]  rows {roff}–{roff+npr}/{page['nrows']}   "
                   f"cols {coff}–{coff+npc}/{page['ncols']}   "
                   f"[dim][ ] rows   {{ }} cols   esc close[/]")

    async def action_grid_page(self, axis, direction):
        if not self._grid: return
        acc, roff, coff = self._grid
        if axis == 'rows': roff = max(0, roff + direction*_PAGE)
        else:              coff = max(0, coff + direction*_PAGE)
        self._grid = [acc, roff, coff]; await self._render_grid()

    async def action_toggle_grid(self):
        w = self.query_one('#gridwrap')
        if w.has_class('visible'): w.remove_class('visible'); return
        n = self._tree.cursor_node
        d = (n.data or {}) if n else {}
        if d.get('kind') == 'grid': await self._open_grid(d['v'])
        elif d.get('v', {}).get('is_grid'): await self._open_grid(d['v'])

    def action_hide_panels(self):
        "esc: close the autocomplete popup, else the exec bar, else the grid panel."
        if self._ac_visible: self._ac_close(); return
        ew = self.query_one('#execwrap')
        if ew.has_class('visible'): ew.remove_class('visible'); self._tree.focus(); return
        self.query_one('#gridwrap').remove_class('visible')

    async def action_profile(self, name):
        self.profile = name; await self._reload()

    async def action_refresh(self): await self._reload()

    async def action_share_selected(self):
        "Push the selected variable into the agent's isolated worker as a read-only copy."
        acc = self._cursor_acc()
        if not acc: self.notify('select a variable first'); return
        name = acc[0]
        try: await self.client.share(name); self.notify(f'shared {name} → agent (read-only copy)')
        except Exception as e: self.notify(f'share failed: {e}', severity='error')

    async def action_share_all(self):
        "Push read-only copies of all owner variables into the agent's isolated worker."
        try: await self.client.share_all(); self.notify('shared all vars → agent (read-only copies)')
        except Exception as e: self.notify(f'share all failed: {e}', severity='error')

    # -- filter + exec --------------------------------------------------------
    def action_focus_filter(self):
        self.query_one('#filter', Input).focus()

    def _rebuild_filtered(self):
        if self._data is not None: self._build(self._data)

    async def on_input_changed(self, event):
        if event.input.id == 'filter':
            self.filter = (event.value or '').strip().lower(); self._rebuild_filtered()
        elif event.input.id == 'code':
            if self._ac_suppress: self._ac_suppress = False; return
            if self._ac_timer is not None: self._ac_timer.stop()
            self._ac_timer = self.set_timer(0.15, lambda: self.run_worker(self._ac_trigger(), group='ac', exclusive=True))

    async def on_input_submitted(self, event):
        if event.input.id == 'filter':
            self._tree.focus()
        elif event.input.id == 'code':
            code = event.value
            if not code.strip(): return
            try: r = await self.client.exec_code(code)
            except Exception as e:
                self.query_one('#execout', Static).update(f"[{GRUV['red']}]{e}[/]"); return
            self._show_exec(r); event.input.value = ''
            await self._reload()

    def _show_exec(self, r):
        "Render an /api/exec result dict into the exec output pane."
        g = GRUV; out = []
        res = r.get('result')
        if res: out.append(f"[{g['aqua']}]{res.get('value','')}[/]")
        if r.get('stdout'): out.append(r['stdout'].rstrip())
        if r.get('error'):  out.append(f"[{g['red']}]{r['error']}[/]")
        self.query_one('#execout', Static).update('\n'.join(out) or f"[{g['gray']}]ok[/]")

    async def action_toggle_exec(self):
        w = self.query_one('#execwrap')
        if w.has_class('visible'): w.remove_class('visible'); self._ac_close(); self._tree.focus()
        else: w.add_class('visible'); self.query_one('#code', Input).focus()

    # -- autocomplete (live, via /complete) -----------------------------------
    async def _ac_trigger(self):
        "Query /complete for the token at the cursor and show the popup."
        inp = self.query_one('#code', Input)
        code, pos = inp.value, inp.cursor_position
        if not code[:pos].strip(): self._ac_close(); return
        try: res = await self.client.complete(code, pos)
        except Exception: self._ac_close(); return
        matches = res.get('matches') or []
        if not matches: self._ac_close(); return
        self._ac_matches, self._ac_sel = matches, 0
        self._ac_from = res.get('from', pos)
        self._ac_visible = True; self._ac_render()

    def _ac_render(self):
        g = GRUV
        box = self.query_one('#complist', Static)
        rows = []
        for i, m in enumerate(self._ac_matches[:12]):
            rows.append(f"[{g['bg_h']} on {g['aqua']}] {m} [/]" if i == self._ac_sel else f"[{g['fg2']}] {m}[/]")
        box.update('\n'.join(rows)); box.add_class('visible')

    def _ac_move(self, d):
        if not self._ac_matches: return
        self._ac_sel = (self._ac_sel + d) % len(self._ac_matches); self._ac_render()

    def _ac_accept(self):
        if not self._ac_matches: return
        m = self._ac_matches[self._ac_sel]
        inp = self.query_one('#code', Input)
        code, pos = inp.value, inp.cursor_position
        self._ac_suppress = True
        inp.value = code[:self._ac_from] + m + code[pos:]
        inp.cursor_position = self._ac_from + len(m)
        self._ac_close()

    def _ac_close(self):
        self._ac_visible = False; self._ac_matches = []
        try: self.query_one('#complist', Static).remove_class('visible')
        except Exception: pass

    def _ac_key(self, key):
        if key == 'down': self._ac_move(1)
        elif key == 'up': self._ac_move(-1)
        elif key in ('enter', 'tab'): self._ac_accept()
        elif key == 'escape': self._ac_close()

    # -- env switching --------------------------------------------------------
    @work
    async def action_switch_env(self):
        "Pop the env picker; on choice, reconnect to that env and reload."
        try: envs = await self.client.envs()
        except Exception as e: self._topbar(err=str(e)); return
        if not envs: self._topbar(err='no other envs'); return
        base = await self.push_screen_wait(EnvScreen(envs, self.base))
        self.env = next((e['name'] for e in envs if e['base'] == self.base), self.env)
        if base and base.rstrip('/') != self.base: await self._switch(base)
        else: self._topbar()

    @work
    async def action_sessions(self):
        "Browse saved sessions; load the chosen past cell into the exec bar."
        code = await self.push_screen_wait(SessionScreen(self.client))
        if code:
            self.query_one('#execwrap').add_class('visible')
            inp = self.query_one('#code', Input)
            self._ac_suppress = True
            inp.value = code; inp.cursor_position = len(code); inp.focus()

    async def _switch(self, base):
        "Tear down the current connection/socket and reconnect against a new env base URL."
        for w in list(self.workers):
            if w.name == 'ws': w.cancel()
        try: await self.client.aclose()
        except Exception: pass
        self.client = Client(base); self.base = self.client.base; self.env = None
        await self._reload()
        if self.connect_ws: self.run_worker(self._ws_loop(), name='ws', exclusive=True)

    # -- live socket ----------------------------------------------------------
    async def _ws_loop(self):
        "Hold the /live socket open; any frame means a cell ran -> re-pull. Reconnect on drop."
        from websockets.asyncio.client import connect as ws_connect
        while True:
            try:
                async with ws_connect(self.client.ws_url) as ws:
                    self._set_live(True)
                    async for _ in ws: await self._reload()
            except Exception:
                self._set_live(False)
                await asyncio.sleep(1.0)

    # -- top bar --------------------------------------------------------------
    def _set_live(self, on):
        self._live = on; self._topbar()

    def _topbar(self, err=None):
        try: bar = self.query_one('#topbar', Static)
        except Exception: return
        g = GRUV
        if err:
            state = f"[{g['red']}]○[/] {err[:40]}"
        else:
            state = (f"[{g['green']}]●[/] live" if self._live
                     else f"[{g['yellow']}]○[/] connecting…")
        env = f"  ·  [{g['purple']}]{self.env}[/]" if self.env else ''
        bar.update(f" [b {g['aqua']}]paar[/]{env}  ·  [{g['fg4']}]{self.profile}[/]  ·  {state}"
                   f"    [dim]/ filter   x exec   s sessions   g grid   e env   r refresh   q quit[/]")

In [ ]:
#| export
def _resolve(url=None, env=None):
    "Pick a server base URL: explicit url wins; else discover via the local registry (by env name, or the first live one)."
    if url: return url
    import paar.registry as R
    envs = R.active()
    if not envs: raise SystemExit('no running paar server found - start one with paar.serve() or paar.fasthtml.inspector()')
    if env:
        for e in envs:
            if e['name'] == env: return e['base']
        raise SystemExit(f'env {env!r} not found; live envs: ' + ', '.join(e['name'] for e in envs))
    return envs[0]['base']

def run(url=None, profile='standard', env=None):
    "Launch the terminal inspector. With no url, auto-discover a running paar server from the local registry."
    InspectorApp(base=_resolve(url, env), profile=profile).run()

def main(argv=None):
    "Console entry point: `paar-tui [--url ...] [--env ...] [--profile ...]`."
    p = argparse.ArgumentParser(prog='paar-tui', description='live terminal namespace inspector')
    p.add_argument('--url', default=None, help='paar server base URL (default: discover from the local registry)')
    p.add_argument('--env', default=None, help='connect to this env name from the registry')
    p.add_argument('--profile', default='standard', choices=['minimal', 'standard', 'full'])
    a = p.parse_args(argv); run(a.url, a.profile, a.env)

In [ ]:
from types import SimpleNamespace

def test_ws_url():
    assert _ws_url('http://127.0.0.1:8000') == 'ws://127.0.0.1:8000/live'
    assert _ws_url('https://h:9/') == 'wss://h:9/live'

def test_acc_roundtrip():
    assert json.loads(__import__('urllib.parse', fromlist=['unquote']).unquote(_acc(('a', 3)))) == ['a', 3]

def test_fmt_row_basic():
    t = fmt_row({'name': 'x', 'type': 'int', 'value': '5', 'accessor': ['x']})
    s = t.plain
    assert 'x' in s and '{int}' in s and '5' in s

def test_fmt_row_shape_dtype_grid():
    t = fmt_row({'name': 'arr', 'type': 'ndarray', 'shape': '(3,)', 'dtype': 'int64',
                 'value': '[1 2 3]', 'is_grid': True, 'accessor': ['arr']})
    s = t.plain
    assert 'ndarray: (3,)' in s and 'int64' in s and '\u25a6' in s   # shape, dtype, grid mark

def test_fmt_row_truncates():
    t = fmt_row({'name': 'big', 'type': 'str', 'value': 'z'*500, 'accessor': ['big']}, maxval=20)
    assert t.plain.endswith('\u2026') and len(t.plain) < 120

def test_theme_is_dark_gruvbox():
    th = gruvbox_theme()
    assert th.dark and th.background == GRUV['bg_h'] and th.primary == GRUV['aqua']

for _t in [test_ws_url, test_acc_roundtrip, test_fmt_row_basic, test_fmt_row_shape_dtype_grid,
           test_fmt_row_truncates, test_theme_is_dark_gruvbox]: _t()

In [ ]:
class _StubClient:
    "In-memory stand-in for Client so the app can be driven with no server."
    base = 'http://stub'
    ws_url = 'ws://stub/live'
    async def rows(self, profile=None):
        return {'profile': profile or 'standard', 'profiles': ['minimal', 'standard', 'full'],
                'groups': [{'label': None, 'nodes': [
                    {'name': 'alpha', 'type': 'int', 'value': '123', 'accessor': ['alpha'],
                     'is_container': False, 'is_grid': False},
                    {'name': 'd', 'type': 'dict', 'value': '{...}', 'accessor': ['d'],
                     'is_container': True, 'is_grid': False},
                    {'name': 'df', 'type': 'DataFrame', 'value': '...', 'accessor': ['df'],
                     'is_container': True, 'is_grid': True}]},
                    {'label': 'Modules', 'nodes': [
                    {'name': 'mymod', 'type': 'module', 'value': 'math', 'accessor': ['mymod'],
                     'is_container': True, 'is_grid': False}]}]}
    async def expand(self, accessor, offset=0):
        return [{'name': "'x'", 'type': 'int', 'value': '1', 'accessor': list(accessor)+[0],
                 'is_container': False, 'is_grid': False}]
    async def grid(self, accessor, roff=0, coff=0): return None
    async def envs(self): return [{'name': 'a', 'base': 'http://stub', 'port': 1},
                                  {'name': 'b', 'base': 'http://other', 'port': 2}]
    async def sessions(self):
        return {'current': 'session_A', 'sessions': [
            {'name': 'session_A', 'count': 1, 'current': True},
            {'name': 'session_B', 'count': 2, 'current': False}]}
    async def session(self, name):
        return {'name': name, 'cells': ['x = 1', 'y = x + 1']}
    async def complete(self, code, pos):
        return {'from': 0, 'matches': ['math', 'map', 'max']}
    async def exec_code(self, code, scope='global'):
        return {'ok': True, 'result': {'name': 'result', 'type': 'int', 'value': '5'}, 'stdout': '', 'error': None}
    async def share(self, name): return 'ok'
    async def share_all(self): return 'ok'
    async def aclose(self): pass

async def _pilot():
    app = InspectorApp(client=_StubClient(), connect_ws=False)
    async with app.run_test() as pilot:
        labels = [str(n.label) for n in app._tree.root.children]
        assert any('alpha' in l for l in labels), labels          # data node rendered
        assert any('MODULES' in l for l in labels), labels        # group heading rendered
        dnode = app._by_acc[('d',)]
        dnode.expand(); await pilot.pause()
        assert dnode.children, 'container lazily loaded its child on expand'
        gnode = app._by_acc[('df',)]                              # gridable container
        gnode.expand(); await pilot.pause()
        kinds = [(c.data or {}).get('kind') for c in gnode.children]
        assert 'grid' in kinds, kinds                             # grid toggle child added under a gridable container
        assert [e['name'] for e in await app.client.envs()] == ['a', 'b']   # env discovery feed
        app.filter = 'alph'; app._rebuild_filtered(); await pilot.pause()      # name filter
        tops = [str(n.label) for n in app._tree.root.children]
        assert any('alpha' in l for l in tops) and not any('MODULES' in l for l in tops), tops
        app.filter = ''; app._rebuild_filtered(); await pilot.pause()
        res = await app.client.exec_code('2+3')                                # code-exec feed
        assert res['ok'] and res['result']['value'] == '5'
        await app.action_toggle_exec(); await pilot.pause()                    # autocomplete
        inp = app.query_one('#code', Input); inp.value = 'ma'; inp.cursor_position = 2
        await app._ac_trigger(); await pilot.pause()
        assert app._ac_visible and app._ac_matches, app._ac_matches
        app._ac_accept()
        assert app.query_one('#code', Input).value == 'math'                   # completion inserted
        scr = SessionScreen(app.client)                                        # sessions browser
        await app.push_screen(scr); await pilot.pause()
        labels = [str(o.prompt) for o in scr.query_one('#sesslist', OptionList).options]
        assert any('session_A' in l for l in labels) and any('session_B' in l for l in labels), labels
        await scr._show_cells('session_A'); await pilot.pause()
        cells = [str(o.prompt) for o in scr.query_one('#sesslist', OptionList).options]
        assert any('x = 1' in l for l in cells), cells
        await app.pop_screen()
    await app.client.aclose()

def test_resolve_registry():
    import os, tempfile, paar.registry as R
    os.environ['PAAR_REG_DIR'] = tempfile.mkdtemp()
    R.register('envx', 7777)
    try:
        assert _resolve().endswith(':7777')            # discovers the sole live env
        assert _resolve(env='envx').endswith(':7777')  # by name
        assert _resolve('http://explicit') == 'http://explicit'   # explicit url wins
    finally: R.unregister(7777)
test_resolve_registry()

try: asyncio.get_running_loop()
except RuntimeError: asyncio.run(_pilot())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()